# Notebook 67 — Policy Learning

**Policy learning specifies adaptive controller candidates.**

Notebook 67 continues the learning-controller arc opened by Notebook 61. The goal is not immediate runtime control. The goal is to learn candidate policies from validated telemetry, rank them offline, and pass only bounded controller candidates forward.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import zipfile, json

OUT = Path("notebook_67_v2_outputs")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 34,
    "axes.labelsize": 24,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 15,
})

In [2]:
def rounded_box(ax, xy, width, height, text, fontsize=16, lw=2.0):
    x, y = xy
    box = patches.FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=lw, edgecolor="black", facecolor="white"
    )
    ax.add_patch(box)
    ax.text(x + width/2, y + height/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def arrow(ax, start, end, lw=2.0):
    ax.annotate(
        "", xy=end, xytext=start,
        arrowprops=dict(arrowstyle="->", lw=lw, color="black", shrinkA=4, shrinkB=4)
    )

def savefig(name):
    path = OUT / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    return path

## 1. Notebook 67 position

Notebook 61 produced validated telemetry records. Notebook 67 uses those records to learn candidate policies. Later notebooks evaluate, bound, and deploy only after offline checks.

In [3]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Notebook 67 continues the learning-controller arc", pad=20)

top = [
    ("43\nResource\nAllocation", 0.08),
    ("49\nAdaptive\nInfrastructure", 0.30),
    ("53\nDistributed\nScheduling", 0.52),
    ("59\nCluster\nOptimization", 0.74),
]
for text, x in top:
    rounded_box(ax, (x, 0.68), 0.20, 0.13, text, fontsize=15)
for x in [0.28, 0.50, 0.72]:
    arrow(ax, (x, 0.745), (x+0.02, 0.745))
ax.text(0.50, 0.61, "second arc: controller → infrastructure → distributed scheduling → cluster optimization",
        ha="center", va="center", fontsize=18)
ax.plot([0.08, 0.92], [0.56, 0.56], color="black", lw=2)

bottom = [
    ("61\nTelemetry\nDataset", 0.06, 0.18, 0.18),
    ("67\nPolicy\nLearning", 0.26, 0.18, 0.20),
    ("71\nOffline\nEvaluation", 0.48, 0.18, 0.18),
    ("73\nSafety\nBounds", 0.68, 0.18, 0.18),
    ("79\nAdaptive\nController", 0.88, 0.18, 0.18),
]
for text, x, w, alpha in bottom:
    rounded_box(ax, (x, 0.35), w, 0.13, text, fontsize=15, lw=3 if text.startswith("67") else 2)
for x in [0.24, 0.46, 0.66, 0.86]:
    arrow(ax, (x, 0.415), (x+0.02, 0.415))
ax.text(0.50, 0.28, "third arc: telemetry dataset → policy learning → evaluation → safety → controller",
        ha="center", va="center", fontsize=20)

arrow(ax, (0.84, 0.68), (0.36, 0.48), lw=2.2)
ax.text(0.52, 0.53, "optimized serving traces become training data", ha="center", va="bottom", fontsize=17)

savefig("67_third_arc_position_v2.png")

PosixPath('notebook_67_v2_outputs/67_third_arc_position_v2.png')

## 2. Policy learning pipeline

Validated datasets produce state-action records. Policy learning produces candidates for offline evaluation, not immediate deployment.

In [4]:
fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Learning dataset specifies policy optimization", pad=20)

steps = [
    "validated\ndataset D",
    "state-action\nrecords",
    "policy\nlearner",
    "candidate\npolicy πθ",
    "offline\nscore",
    "controller\ncandidate",
]
y = 0.72
x0 = 0.08
w = 0.14
gap = 0.015
for i, label in enumerate(steps):
    x = x0 + i * (w + gap)
    rounded_box(ax, (x, y-0.08), w, 0.16, label, fontsize=16)
    if i < len(steps)-1:
        arrow(ax, (x+w, y), (x+w+gap, y), lw=2.2)

rounded_box(ax, (0.20, 0.20), 0.60, 0.12,
            "Learning produces candidates for offline evaluation, not immediate deployment.",
            fontsize=18)
savefig("67_learning_pipeline_v2.png")

PosixPath('notebook_67_v2_outputs/67_learning_pipeline_v2.png')

## 3. State vector to policy candidates

The learner maps telemetry-derived state vectors into action scores. The output is a candidate policy, not a live controller.

In [5]:
fig, ax = plt.subplots(figsize=(15, 11))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("State vector maps telemetry into policy candidates", pad=20)

inputs = [
    ("q(t)\nqueue", 0.10, 0.70),
    ("u(t)\nutilization", 0.40, 0.82),
    ("m(t)\nmemory", 0.70, 0.70),
    ("λ(t)\narrival", 0.10, 0.47),
    ("r(t)\nreserve", 0.40, 0.35),
    ("v(t)\nverification", 0.70, 0.47),
]
for label, x, y in inputs:
    rounded_box(ax, (x, y), 0.20, 0.10, label, fontsize=15)

rounded_box(ax, (0.37, 0.57), 0.26, 0.11, "State vector\nS(t)", fontsize=21, lw=2.5)
rounded_box(ax, (0.37, 0.43), 0.26, 0.09, "policy model\nπθ(S)", fontsize=18, lw=2.5)
rounded_box(ax, (0.37, 0.28), 0.26, 0.09, "action scores\nπθ(a | S)", fontsize=18)
rounded_box(ax, (0.37, 0.13), 0.26, 0.09, "candidate action", fontsize=18)

for label, x, y in inputs:
    arrow(ax, (x+0.10, y), (0.50, 0.62), lw=1.7)

arrow(ax, (0.50, 0.57), (0.50, 0.52), lw=2.2)
arrow(ax, (0.50, 0.43), (0.50, 0.37), lw=2.2)
arrow(ax, (0.50, 0.28), (0.50, 0.22), lw=2.2)

ax.text(0.50, 0.06, r"$S(t)=[q(t),u(t),m(t),\lambda(t),r(t),v(t)]$",
        ha="center", va="center", fontsize=25)
savefig("67_policy_model_inputs_v2.png")

PosixPath('notebook_67_v2_outputs/67_policy_model_inputs_v2.png')

## 4. Learning objective

The loss balances imitation or prediction, expected value, constraint penalties, and imbalance penalties.

In [6]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Learning objective balances fit, value, and constraints", pad=20)

steps = [
    "prediction\nloss",
    "value\nloss",
    "constraint\npenalty",
    "imbalance\npenalty",
    "regularized\nobjective",
]
x0, y, w, h = 0.13, 0.62, 0.14, 0.12
for i, label in enumerate(steps):
    x = x0 + i * (w + 0.025)
    rounded_box(ax, (x, y), w, h, label, fontsize=16)
    if i < len(steps)-1:
        arrow(ax, (x+w, y+h/2), (x+w+0.025, y+h/2), lw=2)

ax.text(0.50, 0.38,
        r"$\mathcal{L}(\theta)=\ell(\pi_\theta,\pi_0)-\alpha \widehat{U}(\pi_\theta)+\beta C(\pi_\theta)+\gamma R(\pi_\theta)$",
        ha="center", va="center", fontsize=27)
rounded_box(ax, (0.22, 0.18), 0.56, 0.10,
            "The learner is rewarded for useful policy shifts and penalized for unsafe or unstable ones.",
            fontsize=17)
savefig("67_loss_objective_v2.png")

PosixPath('notebook_67_v2_outputs/67_loss_objective_v2.png')

## 5. Candidate policy surface

Learning converts the labeled dataset into policy regions. The learned policy is intentionally conservative near unsafe adaptation regions.

In [7]:
x = np.linspace(0.05, 0.95, 90)
y = np.linspace(0.05, 0.95, 90)
X, Y = np.meshgrid(x, y)

steady = 0.70 - 0.90*Y + 0.15*np.cos(8*X)
reserve = 0.40*X - 0.25*Y + 0.10*np.sin(5*X)
rebalance = 0.58*Y + 0.50*X - 0.20*np.abs(X-Y)
scaleout = 0.90*X + 0.75*Y - 0.60
shed = 0.95*Y - 1.1*X - 0.20

scores = np.stack([steady, reserve, rebalance, scaleout, shed], axis=0)
policy = np.argmax(scores, axis=0)
labels = ["steady", "reserve", "rebalance", "scale-out", "shed/shorten"]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(policy, origin="lower", extent=[0.05, 0.95, 0.05, 0.95], aspect="auto")
cbar = plt.colorbar(im, ax=ax)
cbar.set_ticks(range(len(labels)))
cbar.set_ticklabels(labels)
cbar.set_label("selected candidate action")

ax.text(0.20, 0.24, "steady", ha="center", va="center", fontsize=20)
ax.text(0.76, 0.25, "reserve", ha="center", va="center", fontsize=20)
ax.text(0.56, 0.57, "rebalance", ha="center", va="center", fontsize=20)
ax.text(0.80, 0.83, "scale-out", ha="center", va="center", fontsize=20)
ax.text(0.18, 0.76, "shed /\nshorten", ha="center", va="center", fontsize=20)

ax.set_title("Learned policy candidate surface")
ax.set_xlabel("reserve capacity")
ax.set_ylabel("active load")
savefig("67_policy_candidate_surface_v2.png")

PosixPath('notebook_67_v2_outputs/67_policy_candidate_surface_v2.png')

## 6. Counterfactual ranking

Candidate policies are ranked offline using expected gain minus latency and spillover costs. The highest-gain action is not necessarily the best policy.

In [8]:
names = ["baseline", "conservative", "balanced", "aggressive", "unsafe"]
gain = np.array([0.52, 0.57, 0.72, 0.83, 0.92])
latency = -np.array([0.25, 0.21, 0.24, 0.38, 0.62])
spillover = -np.array([0.18, 0.14, 0.16, 0.25, 0.48])
net = gain + latency + spillover - np.array([0.06, 0.03, 0.05, 0.16, 0.38])

idx = np.arange(len(names))
width = 0.18

fig, ax = plt.subplots(figsize=(15, 9))
ax.bar(idx - 1.5*width, gain, width, label="gain")
ax.bar(idx - 0.5*width, latency, width, label="latency cost")
ax.bar(idx + 0.5*width, spillover, width, label="spillover cost")
ax.bar(idx + 1.5*width, net, width, label="net score")
ax.axhline(0, color="black", lw=1)
ax.set_xticks(idx)
ax.set_xticklabels(names, rotation=20)
ax.set_ylabel("offline normalized value")
ax.set_title("Counterfactual ranking selects bounded policy candidates")
ax.legend(loc="upper left")
savefig("67_counterfactual_ranking_v2.png")

PosixPath('notebook_67_v2_outputs/67_counterfactual_ranking_v2.png')

## 7. Generalization check

Offline learning requires validation and test checks before policy candidates can move toward deployment.

In [9]:
epochs = np.arange(1, 41)
train = 0.82*np.exp(-epochs/18) + 0.08
valid = 0.76*np.exp(-epochs/20) + 0.17 + 0.015*np.maximum(epochs-24, 0)/16
test = 0.74*np.exp(-epochs/22) + 0.20 + 0.02*np.maximum(epochs-24, 0)/16
selected = 40

fig, ax = plt.subplots(figsize=(15, 9))
ax.plot(epochs, train, label="train loss")
ax.plot(epochs, valid, label="validation loss")
ax.plot(epochs, test, label="test loss")
ax.axvline(selected, ls="--", color="black", label=f"selected epoch {selected}")
ax.set_xlabel("training epoch")
ax.set_ylabel("loss")
ax.set_title("Validation performance bounds policy learning")
ax.legend(loc="lower left")
savefig("67_generalization_gap_v2.png")

PosixPath('notebook_67_v2_outputs/67_generalization_gap_v2.png')

## 8. Constraint regularization

The regularized objective selects a lower-risk adaptation intensity than the unregularized objective.

In [10]:
z = np.linspace(0, 1, 240)
throughput = 1.08*(1 - np.exp(-3.2*z))
latency = 0.18 + 0.75*z**3
risk = 0.10 + 0.12*np.maximum(z - 0.65, 0)**2 * 8
unreg = throughput - latency - 0.6*risk
reg = unreg - 1.35*np.maximum(z - 0.58, 0)**2 - 0.28*np.maximum(0.18 - z, 0)
best = z[np.argmax(reg)]

fig, ax = plt.subplots(figsize=(15, 9))
ax.plot(z, throughput, label="throughput gain")
ax.plot(z, latency, label="latency pressure")
ax.plot(z, risk, label="constraint risk")
ax.plot(z, unreg, label="unregularized objective")
ax.plot(z, reg, lw=3, label="regularized objective")
ax.axvline(best, color="black", ls="--", label=f"selected ≈ {best:.2f}")
ax.set_xlabel("policy adaptation intensity")
ax.set_ylabel("normalized value")
ax.set_title("Constraint regularization avoids unsafe adaptation")
ax.legend(loc="upper left")
savefig("67_constraint_regularization_v2.png")

PosixPath('notebook_67_v2_outputs/67_constraint_regularization_v2.png')

## 9. Final synthesis

Notebook 67 turns the telemetry dataset into a set of bounded policy candidates ready for offline evaluation.

In [11]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Policy learning specifies adaptive controller candidates", pad=20)

steps = [
    "validated\ndataset",
    "state\nwindows",
    "outcome\nlabels",
    "policy\nlearner",
    "candidate\nπθ",
    "offline\nranking",
    "bounded\ncontroller",
]
x0, y, w, h = 0.03, 0.62, 0.13, 0.13
for i, label in enumerate(steps):
    x = x0 + i*(w + 0.012)
    rounded_box(ax, (x, y), w, h, label, fontsize=16, lw=2.2)
    if i < len(steps)-1:
        arrow(ax, (x+w, y+h/2), (x+w+0.012, y+h/2), lw=2.2)

rounded_box(ax, (0.16, 0.22), 0.68, 0.12,
            "Notebook 67 learns candidate policies from telemetry records before runtime control.",
            fontsize=18)
savefig("67_final_synthesis_v2.png")

PosixPath('notebook_67_v2_outputs/67_final_synthesis_v2.png')

In [12]:
# Package generated figures for download.
zip_path = Path("notebook_67_figures_v2.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.glob("67_*_v2.png")):
        zf.write(p, arcname=p.name)

manifest = {
    "notebook": "67_policy_learning_v2",
    "figures": [p.name for p in sorted(OUT.glob("67_*_v2.png"))],
    "zip": str(zip_path),
}
Path("notebook_67_manifest_v2.json").write_text(json.dumps(manifest, indent=2))
manifest

{'notebook': '67_policy_learning_v2',
 'figures': ['67_constraint_regularization_v2.png',
  '67_counterfactual_ranking_v2.png',
  '67_final_synthesis_v2.png',
  '67_generalization_gap_v2.png',
  '67_learning_pipeline_v2.png',
  '67_loss_objective_v2.png',
  '67_policy_candidate_surface_v2.png',
  '67_policy_model_inputs_v2.png',
  '67_third_arc_position_v2.png'],
 'zip': 'notebook_67_figures_v2.zip'}